# Duloxetine liver toxicity starter notebook

This notebook imports functions from the Python package in `duloxetine_liver_toxicity/` so the analysis logic is broken out into reusable modules instead of living entirely in a single notebook cell.

### Workflow
1. Load or create synthetic example data.
2. Build the duloxetine exposure cohort.
3. Define the liver toxicity phenotype.
4. Run a simple baseline model.
5. Inspect outputs.

> Replace the synthetic data below with your ResearchAllOfUs tables or a parquet/csv export.

In [ ]:
import pandas as pd

from duloxetine_liver_toxicity.cohort import build_duloxetine_cohort
from duloxetine_liver_toxicity.phenotype import define_liver_toxicity_outcome
from duloxetine_liver_toxicity.analysis import run_baseline_model

print("Starter notebook loaded successfully.")

In [ ]:
# Create a small synthetic example dataset to demonstrate the workflow.
# Replace this with your actual All of Us extracts later.

drug_exposure = pd.DataFrame(
    {
        "person_id": [1, 1, 2, 3, 4, 5, 6],
        "drug_concept_id": [123456, 123456, 123456, 123456, 999999, 123456, 123456],
        "drug_exposure_start_datetime": pd.to_datetime([
            "2020-01-10", "2021-02-01", "2019-08-20", "2022-03-15",
            "2020-01-01", "2021-05-01", "2021-06-01"
        ]),
    }
)

condition_occurrence = pd.DataFrame(
    {
        "person_id": [2],
        "condition_concept_id": [200],
        "condition_start_datetime": pd.to_datetime(["2018-01-01"]),
    }
)

measurement = pd.DataFrame(
    {
        "person_id": [1, 1, 1, 2, 3, 3, 4, 5, 6],
        "measurement_concept_id": [300, 301, 302, 300, 300, 301, 301, 300, 302],
        "measurement_datetime": pd.to_datetime([
            "2020-01-15", "2020-01-15", "2020-01-15",
            "2019-09-01", "2022-03-20", "2022-03-20",
            "2022-06-01", "2021-05-15", "2021-06-15"
        ]),
        "value_as_number": [110, 90, 2.0, 50, 110, 95, 50, 100, 1.0],
    }
)

person = pd.DataFrame(
    {
        "person_id": [1, 2, 3, 4, 5, 6],
        "year_of_birth": [1980, 1970, 1990, 1985, 1978, 2000],
        "sex_at_birth": ["F", "M", "F", "M", "F", "M"],
        "race": ["White", "Black", "Asian", "White", "Other", "White"],
        "ethnicity": ["Not Hispanic", "Hispanic", "Not Hispanic", "Not Hispanic", "Hispanic", "Not Hispanic"],
    }
)

print("Synthetic example data created.")

In [ ]:
# Build the cohort
cohort = build_duloxetine_cohort(
    drug_exposure=drug_exposure,
    condition_occurrence=condition_occurrence,
    measurement=measurement,
    person=person,
)

print(cohort.head())
print(f"Cohort size: {len(cohort)}")

In [ ]:
# Define the outcome
outcome_cohort = define_liver_toxicity_outcome(
    cohort=cohort,
    measurement=measurement,
    outcome_window_days=365,
    strict_mode=False,
)

print(outcome_cohort[["person_id", "index_date", "liver_toxicity", "outcome_date"]].head())
print(outcome_cohort["liver_toxicity"].value_counts(dropna=False).to_dict())

In [ ]:
# Run a simple baseline model
covariates = [
    "year_of_birth",
    "sex_at_birth",
    "race",
    "ethnicity",
]

metrics = run_baseline_model(
    cohort=outcome_cohort,
    covariates=covariates,
)

print("AUC:", metrics["auc"])
print("Average Precision:", metrics["average_precision"])

## Next steps

- Replace the synthetic inputs with your actual ResearchAllOfUs tables.
- Replace the placeholder concept IDs in `duloxetine_liver_toxicity/config.py`.
- Add genotype/proteomics merging in a dedicated module for later analysis.
- Convert the notebook into a more formal analysis pipeline once the concept mapping is finalized.